# Ready For Kaggle Baseline Reproduction

Strict no-submit reproduction of public Simple fine-tuning baseline. Do not submit from this notebook.

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


In [ ]:
from pathlib import Path
Path('/kaggle/working/scripts').mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile /kaggle/working/scripts/reproduce_detectron2_simple_finetuning_baseline.py
#!/usr/bin/env python3
"""Strict no-submit reproduction of Kaggle public Simple Fine-tuning Baseline.

Protocol:
- Detectron2 DefaultTrainer on unlearn20 with empty annotations
- poisoned_model.pth initialization
- official RetinaNet config and anchors
- LR=1e-4, batch size=4, MAX_ITER=20, STEPS=[]
- full model trainable (no explicit freeze)
- correct 16-bit preprocessing
- DefaultPredictor inference at CONF_THRESH=0.2
- writes local CSV only; never submits
"""
from __future__ import annotations

import argparse
import copy
import csv
import hashlib
import json
import math
import os
import time
from pathlib import Path
from typing import Dict, Iterable, List, Optional

import cv2
import numpy as np
import torch

torch.backends.nnpack.enabled = False
torch.set_num_threads(int(os.environ.get("TORCH_NUM_THREADS", "1")))
torch.set_num_interop_threads(int(os.environ.get("TORCH_NUM_INTEROP_THREADS", "1")))

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, DatasetMapper, MetadataCatalog, build_detection_train_loader, detection_utils as utils
from detectron2.engine import DefaultPredictor, DefaultTrainer
from tqdm import tqdm

ROOT = Path(__file__).resolve().parents[1]
DEFAULT_CKPT = ROOT / "data/raw/poisoned_model.pth"
DEFAULT_UNLEARN_DIR = ROOT / "data/raw"
DEFAULT_TEST_DIR = ROOT / "data/test_download/extracted/test_set/test_set"
DEFAULT_SAMPLE = ROOT / "data/test_download/extracted/sample_submission.csv"
DEFAULT_OUTDIR = ROOT / "artifacts/detectron2_simple_finetuning_baseline_reproduction"
EXPECTED_CKPT_SHA256 = "f6c21faa2a5b56549fc9e058147c90b149a034858fe0678f5a99ea5a6f0e657c"

BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1
UNLEARN_LR = 1e-4
UNLEARN_ITERS = 20
BATCH_SIZE = 4
CONF_THRESH = 0.2
IMG_W = IMG_H = 1024
DATASET_NAME = "neural_debris_unlearn_empty_strict"


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def read_sample(path: Path) -> List[Dict[str, str]]:
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if not rows or list(rows[0].keys()) != ["id", "image_id", "prediction_string"]:
        raise ValueError(f"bad sample schema: {list(rows[0].keys()) if rows else None}")
    return rows


def load_16bit_scaled(path: Path) -> np.ndarray:
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(path)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


def register_unlearn_empty(unlearn_dir: Path) -> List[Dict[str, object]]:
    if DATASET_NAME in DatasetCatalog.list():
        DatasetCatalog.remove(DATASET_NAME)
        MetadataCatalog.remove(DATASET_NAME)
    json_path = unlearn_dir / "annotations_coco.json"
    with open(json_path, encoding="utf-8") as f:
        coco = json.load(f)
    dicts: List[Dict[str, object]] = []
    for im in coco["images"]:
        file_name = unlearn_dir / im["file_name"]
        if not file_name.exists():
            raise FileNotFoundError(file_name)
        dicts.append({
            "file_name": str(file_name),
            "height": im["height"],
            "width": im["width"],
            "image_id": im["id"],
            "annotations": [],
        })
    DatasetCatalog.register(DATASET_NAME, lambda dicts=dicts: dicts)
    MetadataCatalog.get(DATASET_NAME).set(thing_classes=["object"])
    return dicts


class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_16bit_scaled(Path(dataset_dict["file_name"]))
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


class UnlearnTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)


def make_cfg(weights: Path, outdir: Path, *, train: bool, device: str):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = str(weights)
    cfg.MODEL.DEVICE = device
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
    cfg.DATASETS.TRAIN = (DATASET_NAME,) if train else ()
    cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = int(os.environ.get("D2_NUM_WORKERS", "2")) if train else 0
    cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
    cfg.SOLVER.BASE_LR = UNLEARN_LR
    cfg.SOLVER.MAX_ITER = UNLEARN_ITERS
    cfg.SOLVER.STEPS = []
    cfg.OUTPUT_DIR = str(outdir)
    return cfg


def count_trainable_params(model) -> Dict[str, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_params": int(total), "trainable_params": int(trainable), "frozen_params": int(total - trainable)}


def train_baseline(args) -> Path:
    t0 = time.perf_counter()
    args.outdir.mkdir(parents=True, exist_ok=True)
    ckpt_sha = sha256_file(args.checkpoint)
    if ckpt_sha != EXPECTED_CKPT_SHA256:
        raise ValueError(f"checkpoint sha mismatch: {ckpt_sha}")
    dicts = register_unlearn_empty(args.unlearn_dir)
    cfg = make_cfg(args.checkpoint, args.outdir, train=True, device=args.device)
    trainer = UnlearnTrainer(cfg)
    param_counts = count_trainable_params(trainer.model)
    trainer.resume_or_load(resume=False)
    trainer.train()
    final_ckpt = args.outdir / "model_final.pth"
    if not final_ckpt.exists():
        raise FileNotFoundError(final_ckpt)
    audit = {
        "mode": "strict_detectron2_simple_finetuning_baseline_train_no_submit",
        "submitted": False,
        "auto_submit": False,
        "submission_authorized": False,
        "paid_gpu_authorized": False,
        "source_checkpoint": str(args.checkpoint),
        "source_checkpoint_sha256": ckpt_sha,
        "output_checkpoint": str(final_ckpt),
        "output_checkpoint_sha256": sha256_file(final_ckpt),
        "dataset": DATASET_NAME,
        "unlearn_dir": str(args.unlearn_dir),
        "unlearn_images": len(dicts),
        "annotations_empty": True,
        "config": {
            "BASE_CONFIG": BASE_CONFIG,
            "ANCHOR_ASPECT_RATIOS": ANCHOR_ASPECT_RATIOS,
            "ANCHOR_SIZES": ANCHOR_SIZES,
            "NUM_CLASSES": NUM_CLASSES,
            "UNLEARN_LR": UNLEARN_LR,
            "UNLEARN_ITERS": UNLEARN_ITERS,
            "BATCH_SIZE": BATCH_SIZE,
            "CONF_THRESH": CONF_THRESH,
            "SOLVER_STEPS": list(cfg.SOLVER.STEPS),
            "SOLVER_OPTIMIZER": "Detectron2 default SGD",
            "MODEL_DEVICE": cfg.MODEL.DEVICE,
            "DATALOADER_NUM_WORKERS": cfg.DATALOADER.NUM_WORKERS,
        },
        "trainable_scope": "full_detectron2_model_no_explicit_freeze",
        "param_counts_after_build": param_counts,
        "runtime_sec": time.perf_counter() - t0,
    }
    (args.outdir / "strict_train_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    print(json.dumps(audit, indent=2), flush=True)
    return final_ckpt


def pred_string_from_instances(instances) -> str:
    out = instances.to("cpu")
    boxes = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()
    parts: List[str] = []
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, IMG_W))
        y1 = float(np.clip(y1, 0, IMG_H))
        x2 = float(np.clip(x2, 0, IMG_W))
        y2 = float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
        if w == 0 or h == 0:
            continue
        parts.extend([f"{float(s):.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"])
    return " ".join(parts) or " "


def valid_pred_string(s: str) -> Dict[str, object]:
    vals = str(s).strip().split()
    if not vals:
        return {"valid": True, "num_detections": 0, "finite": True}
    ok_group = len(vals) % 5 == 0
    finite = True
    for v in vals:
        try:
            if not math.isfinite(float(v)):
                finite = False
        except Exception:
            finite = False
    return {"valid": ok_group and finite, "num_detections": len(vals) // 5 if ok_group else None, "finite": finite}


def infer_csv(args, checkpoint: Path) -> Path:
    t0 = time.perf_counter()
    args.outdir.mkdir(parents=True, exist_ok=True)
    rows_all = read_sample(args.sample_submission)
    rows = rows_all[args.start_index:]
    if args.max_images:
        rows = rows[: args.max_images]
    if not rows:
        raise ValueError("no rows selected")
    missing = [r["image_id"] for r in rows if not (args.test_dir / f"{r['image_id']}.png").exists()]
    if missing:
        raise FileNotFoundError(f"missing test images count={len(missing)} first={missing[:5]}")
    cfg = make_cfg(checkpoint, args.outdir, train=False, device=args.device)
    cfg.MODEL.WEIGHTS = str(checkpoint)
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
    predictor = DefaultPredictor(cfg)
    suffix = "" if args.start_index == 0 and not args.max_images else f"_part_start{args.start_index}_n{len(rows)}"
    out_csv = args.outdir / f"strict_simple_finetuning_baseline_submission{suffix}.csv"
    audit_path = args.outdir / f"strict_simple_finetuning_baseline_submission{suffix}_audit.json"

    total_det = empty = invalid = nonfinite = 0
    scores_all: List[float] = []
    widths: List[float] = []
    heights: List[float] = []
    top_scores: List[float] = []
    first20 = []
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "image_id", "prediction_string"])
        writer.writeheader()
        for idx, r in enumerate(tqdm(rows, desc=f"Strict baseline inference {args.start_index}+{len(rows)}"), start=1):
            image_id = r["image_id"]
            im = load_16bit_scaled(args.test_dir / f"{image_id}.png")
            out = predictor(im)["instances"].to("cpu")
            pred_s = pred_string_from_instances(out)
            chk = valid_pred_string(pred_s)
            vals = pred_s.strip().split()
            scores = []
            if vals:
                nums = [float(x) for x in vals]
                scores = nums[0::5]
                scores_all.extend(scores)
                widths.extend(nums[3::5])
                heights.extend(nums[4::5])
                top_scores.append(max(scores))
            nd = int(chk["num_detections"] or 0)
            total_det += nd
            empty += int(nd == 0)
            invalid += int(not chk["valid"])
            nonfinite += int(not chk["finite"])
            writer.writerow({"id": r["id"], "image_id": r["image_id"], "prediction_string": pred_s})
            if len(first20) < 20:
                first20.append({"id": r["id"], "image_id": r["image_id"], "num_detections": nd, "top_score": max(scores) if scores else None})
            if idx % 50 == 0:
                f.flush()

    def pct(vals: List[float], q: float):
        if not vals:
            return None
        return float(np.quantile(np.asarray(vals, dtype=np.float64), q))

    audit = {
        "mode": "strict_detectron2_simple_finetuning_baseline_inference_no_submit",
        "submitted": False,
        "checkpoint": str(checkpoint),
        "checkpoint_sha256": sha256_file(checkpoint),
        "sample_submission": str(args.sample_submission),
        "test_dir": str(args.test_dir),
        "output_csv": str(out_csv),
        "output_csv_sha256": sha256_file(out_csv),
        "config": {
            "BASE_CONFIG": BASE_CONFIG,
            "ANCHOR_ASPECT_RATIOS": ANCHOR_ASPECT_RATIOS,
            "ANCHOR_SIZES": ANCHOR_SIZES,
            "NUM_CLASSES": NUM_CLASSES,
            "CONF_THRESH": CONF_THRESH,
            "device": args.device,
            "TEST_DETECTIONS_PER_IMAGE": int(cfg.TEST.DETECTIONS_PER_IMAGE),
            "RETINANET_TOPK_CANDIDATES_TEST": int(cfg.MODEL.RETINANET.TOPK_CANDIDATES_TEST),
            "RETINANET_NMS_THRESH_TEST": float(cfg.MODEL.RETINANET.NMS_THRESH_TEST),
        },
        "row_count_total_sample": len(rows_all),
        "row_start_index": args.start_index,
        "row_count": len(rows),
        "detections_total": total_det,
        "detections_per_image_mean": total_det / len(rows),
        "empty_prediction_images": empty,
        "invalid_prediction_strings": invalid,
        "nonfinite_values": nonfinite,
        "score_distribution": {"count": len(scores_all), "min": pct(scores_all, 0), "p25": pct(scores_all, 0.25), "median": pct(scores_all, 0.5), "p75": pct(scores_all, 0.75), "p95": pct(scores_all, 0.95), "max": pct(scores_all, 1)},
        "top_score_distribution": {"count": len(top_scores), "median": pct(top_scores, 0.5), "p25": pct(top_scores, 0.25), "p75": pct(top_scores, 0.75), "min": pct(top_scores, 0), "max": pct(top_scores, 1)},
        "box_width_distribution": {"count": len(widths), "median": pct(widths, 0.5), "p25": pct(widths, 0.25), "p75": pct(widths, 0.75), "min": pct(widths, 0), "max": pct(widths, 1)},
        "box_height_distribution": {"count": len(heights), "median": pct(heights, 0.5), "p25": pct(heights, 0.25), "p75": pct(heights, 0.75), "min": pct(heights, 0), "max": pct(heights, 1)},
        "per_image_summary_first20": first20,
        "runtime_sec": time.perf_counter() - t0,
    }
    audit_path.write_text(json.dumps(audit, indent=2, default=str), encoding="utf-8")
    print(json.dumps({"csv": str(out_csv), "audit": str(audit_path), "rows": len(rows), "detections_total": total_det, "empty": empty, "invalid": invalid, "nonfinite": nonfinite, "runtime_sec": audit["runtime_sec"]}, indent=2), flush=True)
    return out_csv


def main(argv: Optional[Iterable[str]] = None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", type=Path, default=DEFAULT_CKPT)
    ap.add_argument("--unlearn-dir", type=Path, default=DEFAULT_UNLEARN_DIR)
    ap.add_argument("--test-dir", type=Path, default=DEFAULT_TEST_DIR)
    ap.add_argument("--sample-submission", type=Path, default=DEFAULT_SAMPLE)
    ap.add_argument("--outdir", type=Path, default=DEFAULT_OUTDIR)
    ap.add_argument("--device", choices=["cpu", "cuda"], default="cpu")
    ap.add_argument("--train-only", action="store_true")
    ap.add_argument("--infer-only", action="store_true")
    ap.add_argument("--inference-checkpoint", type=Path, default=None)
    ap.add_argument("--start-index", type=int, default=0)
    ap.add_argument("--max-images", type=int, default=0)
    args = ap.parse_args(list(argv) if argv is not None else None)
    if args.train_only and args.infer_only:
        raise ValueError("choose at most one of --train-only/--infer-only")
    ckpt = args.inference_checkpoint
    if not args.infer_only:
        ckpt = train_baseline(args)
    if not args.train_only:
        if ckpt is None:
            ckpt = args.outdir / "model_final.pth"
        infer_csv(args, ckpt)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/scripts/audit_detectron2_simple_finetuning_baseline.py
#!/usr/bin/env python3
"""Merge and audit strict Detectron2 Simple Fine-tuning Baseline CSV shards.
No submission; structure/statistics only.
"""
from __future__ import annotations
import argparse, csv, hashlib, json, math, time
from pathlib import Path
from typing import Dict, List
import numpy as np

ROOT = Path(__file__).resolve().parents[1]
DEFAULT_SAMPLE = ROOT / "data/test_download/extracted/sample_submission.csv"
DEFAULT_OUTDIR = ROOT / "artifacts/detectron2_simple_finetuning_baseline_reproduction"

PUBLIC_REFERENCES = {
    "poisoned_model_reference": {"public_score": 378.0181, "detections_total": 2593, "detections_per_image_mean": 1.2965, "empty_prediction_images": 634},
    "empty_submission_reference": {"public_score": 284.2028, "detections_total": 0, "detections_per_image_mean": 0.0, "empty_prediction_images": 2000},
    "simple_fine_tuning_baseline": {"public_score": 259.9160},
    "repair_suppress_001_submitted": {"public_score": 449.9513, "detections_total": 1160, "detections_per_image_mean": 0.58, "empty_prediction_images": 1132},
}

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_csv(path: Path) -> List[Dict[str,str]]:
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def write_csv(path: Path, rows: List[Dict[str,str]]):
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['id','image_id','prediction_string'])
        w.writeheader(); w.writerows(rows)

def parse_rows(rows):
    det_counts=[]; scores=[]; top_scores=[]; widths=[]; heights=[]; invalid=0; nonfinite=0; empty=0
    for r in rows:
        vals = str(r.get('prediction_string','')).strip().split()
        if not vals:
            det_counts.append(0); empty += 1; continue
        if len(vals) % 5 != 0:
            invalid += 1; det_counts.append(None); continue
        try:
            nums = [float(x) for x in vals]
        except Exception:
            invalid += 1; det_counts.append(None); continue
        if not all(math.isfinite(x) for x in nums):
            nonfinite += 1
        sc=nums[0::5]; ws=nums[3::5]; hs=nums[4::5]
        det_counts.append(len(sc)); scores.extend(sc); widths.extend(ws); heights.extend(hs); top_scores.append(max(sc))
    return dict(det_counts=det_counts, scores=scores, top_scores=top_scores, widths=widths, heights=heights, invalid=invalid, nonfinite=nonfinite, empty=empty)

def dist(vals):
    vals=[v for v in vals if v is not None]
    if not vals:
        return {'count':0,'min':None,'p25':None,'median':None,'p75':None,'p95':None,'max':None}
    a=np.asarray(vals, dtype=float)
    return {'count':int(a.size),'min':float(np.quantile(a,0)),'p25':float(np.quantile(a,.25)),'median':float(np.quantile(a,.5)),'p75':float(np.quantile(a,.75)),'p95':float(np.quantile(a,.95)),'max':float(np.quantile(a,1))}

def summarize(rows):
    p=parse_rows(rows)
    valid_counts=[x for x in p['det_counts'] if x is not None]
    return {
        'row_count': len(rows),
        'detections_total': int(sum(valid_counts)),
        'detections_per_image_mean': float(sum(valid_counts)/len(rows)) if rows else None,
        'empty_prediction_images': int(p['empty']),
        'invalid_prediction_strings': int(p['invalid']),
        'nonfinite_values': int(p['nonfinite']),
        'score_distribution': dist(p['scores']),
        'top_score_distribution': dist(p['top_scores']),
        'box_width_distribution': dist(p['widths']),
        'box_height_distribution': dist(p['heights']),
        'detections_per_image_distribution': dist(valid_counts),
        'confidence_sum': float(sum(p['scores'])),
        'confidence_mean_per_image': float(sum(p['scores'])/len(rows)) if rows else None,
    }

def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--outdir', type=Path, default=DEFAULT_OUTDIR)
    ap.add_argument('--sample-submission', type=Path, default=DEFAULT_SAMPLE)
    ap.add_argument('--generated-csv', type=Path, default=None)
    ap.add_argument('--shards', nargs='*', type=Path)
    args=ap.parse_args()
    t0=time.perf_counter(); args.outdir.mkdir(parents=True, exist_ok=True)
    sample=read_csv(args.sample_submission)
    sample_ids=[r['id'] for r in sample]
    sample_image_ids=[r['image_id'] for r in sample]
    out_csv = args.generated_csv or (args.outdir/'strict_simple_finetuning_baseline_submission.csv')
    shard_paths=[]
    if out_csv.exists() and not args.shards:
        merged=read_csv(out_csv)
    else:
        shard_paths=args.shards or sorted(args.outdir.glob('strict_simple_finetuning_baseline_submission_part_start*_n*.csv'))
        rows_by_id={}
        for sp in shard_paths:
            for r in read_csv(sp):
                if r['id'] in rows_by_id:
                    raise ValueError(f'duplicate id {r["id"]} from {sp}')
                rows_by_id[r['id']]=r
        missing=[sid for sid in sample_ids if sid not in rows_by_id]
        extra=[sid for sid in rows_by_id if sid not in set(sample_ids)]
        if missing or extra:
            raise ValueError(f'bad coverage missing={len(missing)} extra={len(extra)} first_missing={missing[:5]} first_extra={extra[:5]}')
        merged=[{'id':r['id'], 'image_id':r['image_id'], 'prediction_string':rows_by_id[r['id']]['prediction_string']} for r in sample]
        write_csv(out_csv, merged)
    gen_summary=summarize(merged); sample_summary=summarize(sample)
    structure={
        'columns_generated': list(merged[0].keys()) if merged else [],
        'columns_sample': list(sample[0].keys()) if sample else [],
        'same_row_count': len(merged)==len(sample),
        'same_id_order': [r['id'] for r in merged]==sample_ids,
        'same_image_id_order': [r['image_id'] for r in merged]==sample_image_ids,
        'csv_valid_for_local_no_submit': bool(len(merged)==len(sample) and [r['id'] for r in merged]==sample_ids and [r['image_id'] for r in merged]==sample_image_ids and gen_summary['invalid_prediction_strings']==0 and gen_summary['nonfinite_values']==0),
    }
    comp={
        'mode':'strict_detectron2_simple_finetuning_baseline_comparison_no_submit',
        'submitted':False,
        'generated_csv':str(out_csv),
        'generated_csv_sha256':sha256_file(out_csv),
        'sample_submission':str(args.sample_submission),
        'sample_submission_sha256':sha256_file(args.sample_submission),
        'shards':[str(p) for p in shard_paths],
        'structure':structure,
        'generated_summary':gen_summary,
        'poisoned_sample_summary':sample_summary,
        'delta_generated_minus_poisoned_sample':{
            'detections_total': gen_summary['detections_total']-sample_summary['detections_total'],
            'detections_per_image_mean': gen_summary['detections_per_image_mean']-sample_summary['detections_per_image_mean'],
            'empty_prediction_images': gen_summary['empty_prediction_images']-sample_summary['empty_prediction_images'],
            'confidence_sum': gen_summary['confidence_sum']-sample_summary['confidence_sum'],
            'score_median': (gen_summary['score_distribution']['median']-sample_summary['score_distribution']['median']) if gen_summary['score_distribution']['median'] is not None and sample_summary['score_distribution']['median'] is not None else None,
        },
        'public_references': PUBLIC_REFERENCES,
        'qualitative_position': 'computed after audit: compare output sparsity/confidence to poisoned, empty, repair-suppress and simple public baseline score reference; no leaderboard score because no-submit',
        'runtime_sec': time.perf_counter()-t0,
    }
    out_audit=args.outdir/'strict_simple_finetuning_baseline_comparison_audit.json'
    out_audit.write_text(json.dumps(comp, indent=2), encoding='utf-8')
    print(json.dumps({'csv':str(out_csv),'audit':str(out_audit),'structure':structure,'generated':gen_summary,'poisoned_sample':sample_summary,'delta':comp['delta_generated_minus_poisoned_sample']}, indent=2))
    return 0 if structure['csv_valid_for_local_no_submit'] else 2

if __name__ == '__main__':
    raise SystemExit(main())


In [ ]:
# Strict baseline reproduction: no submit, no sweep, no variant.
# Paths match public notebook audit.
import os, subprocess, sys
from pathlib import Path

os.environ.setdefault("TORCH_NUM_THREADS", "2")
os.environ.setdefault("TORCH_NUM_INTEROP_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("D2_NUM_WORKERS", "2")

OUT = Path("/kaggle/working/artifacts/detectron2_simple_finetuning_baseline_reproduction")
OUT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "/kaggle/working/scripts/reproduce_detectron2_simple_finetuning_baseline.py",
    "--device", "cuda",
    "--checkpoint", "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth",
    "--unlearn-dir", "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/unlearn_set",
    "--test-dir", "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/test_set/test_set",
    "--sample-submission", "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/sample_submission.csv",
    "--outdir", str(OUT),
]
print("RUN", " ".join(cmd))
with open(OUT/"run_stdout.log", "w") as so, open(OUT/"run_stderr.log", "w") as se:
    proc = subprocess.run(cmd, stdout=so, stderr=se, text=True)
print("exit_code", proc.returncode)
if proc.returncode != 0:
    print((OUT/"run_stderr.log").read_text()[-4000:])
    raise SystemExit(proc.returncode)


In [ ]:
# Build SHA256 manifest for recovery on VPS.
from pathlib import Path
import hashlib, json, time, os
OUT = Path("/kaggle/working/artifacts/detectron2_simple_finetuning_baseline_reproduction")
paths = [
    Path("/kaggle/working/scripts/reproduce_detectron2_simple_finetuning_baseline.py"),
    Path("/kaggle/working/scripts/audit_detectron2_simple_finetuning_baseline.py"),
    Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth"),
    OUT/"model_final.pth",
    OUT/"last_checkpoint",
    OUT/"strict_train_audit.json",
    OUT/"strict_simple_finetuning_baseline_submission.csv",
    OUT/"strict_simple_finetuning_baseline_submission_audit.json",
    OUT/"run_stdout.log",
    OUT/"run_stderr.log",
]
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda:f.read(1024*1024), b''):
            h.update(c)
    return h.hexdigest()
manifest = {
    "created_at_utc": time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    "mode": "kaggle_notebook_strict_public_simple_finetuning_baseline_no_submit",
    "submitted": False,
    "auto_submit": False,
    "submission_authorized": False,
    "paid_gpu_authorized": False,
    "hyperparams": {"UNLEARN_LR":1e-4,"UNLEARN_ITERS":20,"BATCH_SIZE":4,"CONF_THRESH":0.2,"BASE_CONFIG":"COCO-Detection/retinanet_R_50_FPN_3x.yaml"},
    "files": []
}
for p in paths:
    manifest["files"].append({"path": str(p), "exists": p.exists(), "size": p.stat().st_size if p.exists() else None, "sha256": sha256_file(p) if p.exists() and p.is_file() else None})
(OUT/"sha256_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))


Expected outputs under `/kaggle/working/artifacts/detectron2_simple_finetuning_baseline_reproduction/`: `model_final.pth`, CSV, audit JSON, logs, `sha256_manifest.json`.